# 🛰️ Production Roadmap & Research Notebook
## SmartMine Vision AI — from prototype to production

> **Status:** `draft` · research scaffold. This is a *living* document of
> **concepts and tools to investigate**, not a finished design. Each section
> states the problem, the options worth researching, and concrete next steps.
> Code cells are light and defensive — they introspect the repo and the trained
> baseline so the notebook runs end-to-end, but the value is in the prose.

---

### How to use this notebook
- Read top-to-bottom: it follows the path a frame takes from a camera to an alert.
- Each block ends with **🔬 To investigate** (tools/papers) and **✅ Next step**
  (something concrete, often "write a spec").
- The **§9 backlog** aggregates every next step into a prioritized table.

### Scope anchor
The commercial proposal targets **mAP > 0.70**, real-time FPS, and fast response
times (`docs/SmartMine_Vision_AI_Project_Proposal.md`). Per the constitution
those are *vision*, not acceptance criteria — concrete criteria belong in specs.

## 1. Current state vs. target

In [ ]:
import sys
from pathlib import Path

# Resolve project root by walking up until 'src/' is found
ROOT = Path().resolve()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

In [ ]:
# Introspect what exists today so the roadmap is grounded in the real repo.
nb_root = ROOT / "notebooks"
phases = sorted(p.name for p in nb_root.iterdir()
                if p.is_dir() and p.name[:2].isdigit())
print("Pipeline phases (notebooks/):")
for p in phases:
    n = len(list((nb_root / p).glob("*.ipynb")))
    print(f"  {p:<24} {n} notebook(s)")

print("\nsrc/ modules:")
for m in sorted(p.name for p in (ROOT / "src").iterdir() if p.is_dir()):
    print(f"  - {m}")

print("\nModel weights present:")
mdl = ROOT / "models"
found = False
for w in sorted(mdl.rglob("*.pt")):
    print(f"  {w.relative_to(ROOT)}  ({w.stat().st_size/1e6:.1f} MB)")
    found = True
if not found:
    print("  (none yet — run notebooks/01_ppe_detection/03_training_yolo.ipynb)")

**Where we are:** a reproducible *offline* pipeline — explore → validate → train
→ evaluate → image/video inference — driven by notebooks generated from
`scripts/generate_notebooks.py`, with domain logic in `src/ppe_detection/`.

**Where we are going:** an *online* multi-camera service that ingests RTSP
streams, detects PPE/vehicles, tracks workers, reasons about proximity, raises
alerts, and feeds a safety dashboard — reliably, at the edge of a mine with poor
connectivity.

The gap is not "a better model". It is **everything around the model**: serving,
streaming, tracking, alerting, data flywheel, MLOps, and governance. The rest of
this notebook is that gap, broken into researchable pieces.

## 2. Target architecture — from notebooks to a streaming service

A camera frame should flow through decoupled stages so each can scale and fail
independently:

```
        ┌─────────┐   RTSP    ┌──────────────┐   frames   ┌───────────────┐
        │ IP cams │ ────────▶ │  Ingest /     │ ─────────▶ │  Inference     │
        │ (site)  │           │  decode       │            │  service       │
        └─────────┘           │ (GStreamer/   │            │ (YOLO + PPE    │
                              │  DeepStream)  │            │  compliance)   │
                               └──────────────┘            └──────┬────────┘
                                                                  │ detections
                          ┌───────────────────────────────────────┼────────────┐
                          ▼                       ▼                ▼            ▼
                    ┌───────────┐         ┌──────────────┐  ┌───────────┐ ┌──────────┐
                    │ Tracking  │────────▶│  Proximity / │  │  Event    │ │ Frame /  │
                    │ (ByteTrack│ tracks  │  rule engine │─▶│  bus      │ │ clip     │
                    │  /BoT-SORT│         │ (alert logic)│  │(MQTT/Kafka│ │ storage  │
                    └───────────┘         └──────────────┘  └─────┬─────┘ └──────────┘
                                                                  ▼
                                                      ┌────────────────────┐
                                                      │ DB + Dashboard /    │
                                                      │ alert sinks (Power  │
                                                      │ BI, Slack, sirens)  │
                                                      └────────────────────┘
```

This mirrors the repo's own phase layout (`tracking`, `proximity`, `database`,
`api`, `powerbi`) — the modules already anticipate this shape.

**Edge vs. cloud.** Mines have intermittent uplinks, so inference should run
**on-site (edge)**: a small GPU box or edge accelerator per cluster of cameras,
with only events/clips synced to the cloud. The cloud does training, model
registry, fleet monitoring, and long-term storage.

- **🔬 To investigate:** NVIDIA **DeepStream** (multi-stream decode+inference on
  one GPU), **GStreamer** pipelines, **RTSP** restream (mediamtx), edge hardware
  (Jetson Orin, Hailo-8, Google Coral), **MQTT** vs **Kafka** for the event bus.
- **✅ Next step:** write a spec `0XX-streaming-inference` defining the ingest →
  inference → event-bus contract (one camera, one event schema) before building.

## 3. Model serving & inference optimization

A `.pt` model called from Python per frame will not hit real-time on multi-cam.
Production inference needs an **export + runtime** path and **batching**.

| Lever | What it buys | Tool to investigate |
|---|---|---|
| Export to **ONNX** | portable graph, faster CPU | `model.export(format="onnx")`, `onnxruntime` |
| **OpenVINO** | 2–4× on Intel CPU/iGPU | Intel OpenVINO |
| **TensorRT** | best NVIDIA GPU latency, INT8 | TensorRT, `format="engine"` |
| **CoreML** | Apple edge / on-device | `format="coreml"` |
| **INT8 quantization** | ~2–4× speed, small mAP drop | TensorRT/OpenVINO PTQ, calibration set |
| **Server + batching** | many cameras per GPU | **NVIDIA Triton**, dynamic batching |
| **Half precision** | ~2× on GPU | `half=True` at export/infer |

The cell below exports the trained baseline to ONNX and measures a CPU latency
baseline — your starting point before any optimization.

In [ ]:
weights = ROOT / "models" / "ppe" / "yolov8n_smartmine_baseline.pt"
print("Trained baseline present:", weights.exists())

onnx_path = None
if weights.exists():
    try:
        from ultralytics import YOLO
        onnx_path = YOLO(str(weights)).export(format="onnx", imgsz=416)
        print("Exported ONNX ->", onnx_path)
    except Exception as e:
        print(f"ONNX export skipped ({type(e).__name__}: {str(e)[:120]}).")
        print("Install the exporter with: pip install onnx onnxslim onnxruntime")
else:
    print("Run nb03 (training) first, then re-run this cell.")

In [ ]:
# CPU latency micro-benchmark (PyTorch path). Run when the CPU is otherwise idle.
import time
import numpy as np
if weights.exists():
    try:
        from ultralytics import YOLO
        m = YOLO(str(weights))
        dummy = np.random.randint(0, 255, (416, 416, 3), dtype=np.uint8)
        for _ in range(3):                       # warm-up
            m.predict(dummy, imgsz=416, verbose=False)
        N, t0 = 20, time.perf_counter()
        for _ in range(N):
            m.predict(dummy, imgsz=416, verbose=False)
        dt = (time.perf_counter() - t0) / N
        print(f"PyTorch CPU: {dt*1000:6.1f} ms/frame  (~{1/dt:4.1f} FPS, single stream)")
        print("Compare against ONNX Runtime / OpenVINO once exported — that delta")
        print("is your cheapest path to more cameras-per-box.")
    except Exception as e:
        print(f"Benchmark skipped ({type(e).__name__}: {str(e)[:120]}).")
else:
    print("No trained model yet — skipping benchmark.")

- **🔬 To investigate:** Triton Inference Server (model repo, dynamic batching,
  ensemble), ONNX Runtime execution providers, OpenVINO, TensorRT INT8 with a
  calibration set drawn from the mine's own footage.
- **✅ Next step:** spec an export+serve benchmark — measure mAP and FPS for
  `pt → onnx → openvino/tensorrt (fp16/int8)` and pick the deployment target.

## 4. The data flywheel & dataset quality

The model is only as good as the labels. `02_dataset_validation.ipynb` already
surfaced a concrete issue worth triaging.

In [ ]:
import json
rep = ROOT / "docs" / "research" / "smartmine_validation_report.json"
if rep.exists():
    d = json.loads(rep.read_text())
    print("Validation verdict:", d.get("verdict"))
    print("Real issue counts:")
    for k, v in d.get("real_issue_counts", {}).items():
        print(f"  {k[:60]:<60} {v}")
    print("\nNote: 'missing_label' = image with NO .txt file. Ultralytics treats")
    print("those as *background* images (no objects), but they may also be lost")
    print("labels from the merge step — they need a human triage, not a silent pass.")
else:
    print("Run nb02 (validation) to generate the report first.")

**Findings to act on**
- **~640 images without label files.** Are they intentional backgrounds, or did
  `scripts/merge_datasets.py` drop labels? Spot-check against `datasets/raw/`.
  Backgrounds are *useful* (suppress false positives) — but only if intentional.
- **Duplicates** across sources inflate metrics and leak between splits. Dedup by
  perceptual hash, then re-split.

**Building the flywheel** (each deployment makes the next model better):
1. **Capture** hard frames in production (low-confidence, disagreements, alerts).
2. **Auto-label** with the current model, **human-correct** the hard ones.
3. **Version** data + splits; retrain; evaluate; promote.

- **🔬 To investigate:** **CVAT** / **Label Studio** / Roboflow for annotation;
  **DVC** or lakeFS for data versioning; **active learning** (uncertainty/margin
  sampling); perceptual-hash dedup (`imagededup`); split-leakage checks.
- **✅ Next step:** spec a "dataset v2" pipeline: triage missing labels, dedup,
  leakage-safe re-split, and a documented class map.

## 5. Model & task-design improvements

### 5.1 The 32-class schema is doing two jobs at once
The unified schema mixes **embedded compliance** classes
(`person_con_casco`, `person_sin_casco`, …) with **separate PPE objects**
(`hardhat`, `safety_vest`, `mask`). `src/ppe_detection/ppe_classifier.py` already
papers over this at inference time (embedded class wins; separate items fill the
gaps via IoU/centre-containment). That hybrid is fragile and hard to scale to new
PPE types.

**Alternatives to research**
- **Two-stage:** detect `person` once, then a **multi-label attribute** head /
  small classifier per crop (hardhat?/vest?/gloves?…). Decouples "where is the
  worker" from "what are they wearing".
- **Pose-guided association:** use keypoints (head→helmet, torso→vest) to bind PPE
  to the right worker instead of box IoU — robust in crowds/occlusion.

### 5.2 Class imbalance & safety-critical recall
`person` dominates; `mask`/some vehicles are rare. Standard mAP hides what matters:
**a missed `person_sin_casco` (false negative) is a safety failure**, while a false
alarm is mere annoyance. Evaluate with **recall on violation classes** and
**per-condition slices** (dust, low light, distance), not just global mAP.

### 5.3 Model upgrades & domain adaptation
- Try **YOLO11**, **YOLOv8s/m**, or **RT-DETR**; quantize for the edge target.
- Mine-specific nuisances (airborne dust = the `polvo` class, glare, night) →
  targeted augmentation and, if needed, synthetic/Sim data.

- **🔬 To investigate:** multi-label heads, top-down pose (YOLO-pose), focal/class
  weights, **slice-based eval** (e.g. a `polvo`/low-light test split), RT-DETR.
- **✅ Next step:** spec an A/B: current hybrid head vs. person+attribute two-stage,
  scored on **violation recall**, not just mAP.

## 6. MLOps — making training a repeatable, observable process

Today a run lives in `experiments/` with Ultralytics' CSV/plots. That is fine for
one machine; production needs lineage and monitoring.

| Need | Tool to investigate |
|---|---|
| Experiment tracking | **MLflow**, **Weights & Biases**, ClearML |
| Model registry / versioning | MLflow Registry, W&B Artifacts |
| Data/version lineage | **DVC**, lakeFS |
| Pipeline orchestration | Prefect, Dagster, Airflow, Kubeflow |
| CI/CD for models | GitHub Actions + a model-eval gate (mAP/recall thresholds) |
| Production monitoring | **Evidently** (data/prediction drift), Prometheus + Grafana |

**Drift is the silent killer:** seasons, new equipment, new PPE, camera moves —
the input distribution shifts and recall quietly drops. Monitor input statistics
and confidence distributions; trigger retraining when they drift.

- **✅ Next step:** add a CI eval gate (block a model that regresses violation
  recall) and stand up experiment tracking; both are small, high-leverage wins.

## 7. Real-time pipeline: tracking, proximity & multi-camera scale

Detection per-frame is not enough — alerts need **identity over time** and
**spatial reasoning** (the repo's phases 3–4).

- **Tracking** (`src/tracking/`): **ByteTrack** / **BoT-SORT** assign stable IDs so
  an alert fires per *worker-event*, not per *frame* (kills alert spam and enables
  dwell-time rules). Both ship with Ultralytics (`model.track(...)`).
- **Proximity** (`src/proximity/`): person↔vehicle distance needs a ground-plane.
  Research **homography / camera calibration** to map pixels → metres, then
  "person within X m of a moving `volquete`" becomes a real rule.
- **Multi-camera scale:** batch streams on one GPU (DeepStream/Triton), share one
  model, and budget latency end-to-end (decode + infer + track + rule < alert SLA).

- **🔬 To investigate:** ByteTrack/BoT-SORT tuning, homography calibration,
  multi-object-multi-camera (MTMC) re-ID, DeepStream multi-stream batching.
- **✅ Next step:** spec the **alert state machine** (debounce, dwell, cooldown)
  so tracking output becomes trustworthy alerts.

## 8. Safety, privacy & governance

This system watches workers — that carries obligations.

- **Privacy:** faces/identities are personal data. Research **on-edge face
  blurring**, retention limits, and access control. Prefer "event + short clip"
  over "store everything".
- **Human-in-the-loop:** the model *assists* safety officers; keep an override and
  an audit trail. Tune thresholds **per site** to fight alert fatigue.
- **Fairness/robustness:** validate across PPE colours, body types, lighting — a
  detector that only works on the training site's gear is a liability.
- **Governance:** document model cards, data provenance, and decision logs.

- **✅ Next step:** a short "responsible deployment" spec: retention policy,
  anonymization, override/audit requirements.

## 9. Prioritized backlog

Rough **impact × effort**; sequence top-down. "Spec" = create a `specs/###-name/`.

| # | Item | Impact | Effort | Phase | Artifact |
|---|---|---|---|---|---|
| 1 | Triage ~640 missing labels + dedup + leakage-safe re-split | 🔴 High | M | Data | spec: dataset-v2 |
| 2 | Export + serve benchmark (onnx/openvino/tensorrt, fp16/int8) | 🔴 High | M | Serving | spec: inference-bench |
| 3 | Eval by **violation recall** + condition slices (not just mAP) | 🔴 High | S | Model | extend nb04 |
| 4 | Streaming ingest → inference → event bus (1 camera) | 🔴 High | L | Arch | spec: streaming |
| 5 | Tracking (ByteTrack) + alert state machine (debounce/dwell) | 🟠 Med | M | Pipeline | phases 3–4 |
| 6 | Two-stage / attribute head vs. hybrid 32-class (A/B) | 🟠 Med | L | Model | spec: ppe-head-v2 |
| 7 | Experiment tracking + CI eval gate | 🟠 Med | S | MLOps | settings + CI |
| 8 | Drift monitoring (Evidently) on confidence/inputs | 🟡 Low | M | MLOps | monitoring |
| 9 | Privacy/anonymization + retention policy | 🟠 Med | S | Governance | spec: responsible |
| 10 | Proximity via homography (pixels → metres) | 🟡 Low | L | Pipeline | phase 4 |

> Items 1–3 are the cheapest high-impact wins and need no new infrastructure.

## 10. Curated references & tools to investigate

**Serving / edge:** NVIDIA Triton · DeepStream · TensorRT · OpenVINO · ONNX Runtime
· GStreamer · mediamtx · Jetson Orin / Hailo / Coral
**Tracking / geometry:** ByteTrack · BoT-SORT · OpenCV homography & camera
calibration · MTMC re-ID
**Data / labeling:** CVAT · Label Studio · Roboflow · DVC · lakeFS · imagededup ·
active learning (uncertainty sampling)
**MLOps:** MLflow · Weights & Biases · Evidently · Prefect / Dagster · GitHub
Actions model-eval gates
**Models:** Ultralytics YOLO11 / RT-DETR · focal loss & class weighting ·
YOLO-pose for PPE association
**Responsible AI:** model cards · data sheets for datasets · privacy-by-design

---

> **Remember (constitution):** features start as **specs**, not code. Each
> "✅ Next step" above should become a `specs/###-name/spec.md` in `draft` before
> implementation. This notebook is the research that feeds those specs.